# Simulação do Protocolo de Árvore Geradora (STP / RSTP) na Rede SCADA — Aqua Index
## Estudo de Caso: Célula Pneumática de Envasamento de Água em Copos (ECAA08 — Grupo 2)

Este notebook modela a topologia Ethernet industrial (Camada 2) proposta para a implantação em chão de fábrica da célula **Aqua Index**, interligando a estação **SCADA** (`SW-SCADA`), o painel central `PLC-001` (`SW-PLC`) e as três remotas de I/O distribuídas (`SW-IO-IN`, `SW-IO-PROC` e `SW-IO-OUT`) como um grafo não direcionado e ponderado $G = (V, E, w)$.

> **Fronteira arquitetural:** Na simulação local do Automation Studio, a conexão Cliente-Servidor é ponto-a-ponto. Este modelo representa a camada de rede industrial redundante (PROFINET / Modbus TCP) para as remotas de I/O dos 45 sinais ISA-5.1, mantendo o botão de emergência físico `HS-004` (`e1`) fora da rede Ethernet (*hardwired*).

### Objetivos da Simulação
1. **Eleição Determinística do *Root Bridge*** pelo menor $\text{BID} = (\text{Bridge Priority}, \text{MAC Address})$.
2. **Cálculo de *Root Path Cost*** com pesos padrão **IEEE 802.1w (RSTP)** (`10G = 2.000`, `1G = 20.000`, `100M = 200.000`).
3. **Atribuição de Papéis e Estados de Porta** (*Root Port*, *Designated Port*, *Alternate Port / Discarding*) eliminando ciclos L2 (*Broadcast Storms*).
4. **Simulação de Falhas Físicas:**
   - **Falha 1:** Rompimento do backbone principal `SW-SCADA <-> SW-PLC` (10 Gbps) e reconvergência RSTP.
   - **Falha 2:** Queda total do `SW-SCADA` e promoção automática do *Root Bridge* secundário (`SW-PLC`, Priority `8192`).
5. **Comparativo STP vs. RSTP** em ciclos de envase perdidos (*takt* nominal `3,6 s/copo`) e **Auditoria Formal** comprovando $|E_T| = |V_T| - 1$, conectividade e aciclicidade.

### 1. Modelagem do Grafo da Rede SCADA — Aqua Index
O núcleo algorítmico funciona em Python 3 puro (com suporte opcional a `pandas`, `networkx` e `matplotlib` para tabelas e gráficos no Jupyter/Colab).

In [ ]:
from collections import deque
import math

try:
    import pandas as pd
    HAS_PD = True
except ImportError:
    HAS_PD = False

try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

# ==============================================================================
# 1. DEFINIÇÃO DOS VÉRTICES (SWITCHES / REMOTAS DE I/O DA CÉLULA AQUA INDEX)
# ==============================================================================
# Bridge ID (BID) = (priority, mac) -> menor BID vence a eleição
SWITCHES = {
    "SW-SCADA":   {"setor": "Supervisão SCADA & Historiador",       "priority": 4096,  "mac": "00:1A:2B:02:00:01"},
    "SW-PLC":     {"setor": "Painel PLC-001 & Mesa M-001/SE-001",  "priority": 8192,  "mac": "00:1A:2B:02:00:02"},
    "SW-IO-IN":   {"setor": "Remota I/O Entrada (E1 + E2)",         "priority": 32768, "mac": "00:1A:2B:02:00:03"},
    "SW-IO-PROC": {"setor": "Remota I/O Processo (E3 + E4)",        "priority": 32768, "mac": "00:1A:2B:02:00:04"},
    "SW-IO-OUT":  {"setor": "Remota I/O Saída (E5 + EST-500)",      "priority": 32768, "mac": "00:1A:2B:02:00:05"},
}

# ==============================================================================
# 2. DEFINIÇÃO DAS ARESTAS (ENLACES FÍSICOS E CUSTOS IEEE 802.1w)
# ==============================================================================
# Custos padrão RSTP: 10 Gbps = 2.000 | 1 Gbps = 20.000 | 100 Mbps = 200.000
ENLACES = [
    ("SW-SCADA",   "SW-PLC",     {"cost": 2000,   "bandwidth": "10 Gbps (Fibra Backbone)"}),
    ("SW-SCADA",   "SW-IO-OUT",  {"cost": 20000,  "bandwidth": "1 Gbps (Anel Industrial)"}),
    ("SW-PLC",     "SW-IO-IN",   {"cost": 20000,  "bandwidth": "1 Gbps (Anel Industrial)"}),
    ("SW-IO-IN",   "SW-IO-PROC", {"cost": 20000,  "bandwidth": "1 Gbps (Anel Industrial)"}),
    ("SW-IO-PROC", "SW-IO-OUT",  {"cost": 20000,  "bandwidth": "1 Gbps (Anel Industrial)"}),
    # Enlace redundante cruzado entre Painel PLC-001 e Processo (E3/E4)
    ("SW-PLC",     "SW-IO-PROC", {"cost": 200000, "bandwidth": "100 Mbps (Cobre Blindado Backup)"}),
]

def construir_grafo(switches_dict, enlaces_list):
    """Representa G = (V, E, w) como dicionário de adjacência e lista de arestas."""
    nos = {k: dict(v) for k, v in switches_dict.items()}
    adj = {k: {} for k in nos}
    arestas = []
    for u, v, data in enlaces_list:
        if u in nos and v in nos:
            d = dict(data)
            adj[u][v] = d
            adj[v][u] = d
            arestas.append((u, v, d))
    return {"nodes": nos, "adj": adj, "edges": arestas}

G_nominal = construir_grafo(SWITCHES, ENLACES)
n_v = len(G_nominal["nodes"])
n_e = len(G_nominal["edges"])
ciclos_indep = n_e - n_v + 1

print(f"Grafo da Rede SCADA Aqua Index: |V| = {n_v} nós, |E| = {n_e} enlaces físicos.")
print(f"Número ciclomático (|E| - |V| + 1): {ciclos_indep} ciclos independentes (risco de Broadcast Storm sem RSTP).")
assert n_v == 5 and n_e == 6 and ciclos_indep == 2

### 2. Algoritmo RSTP (IEEE 802.1w): Eleição de Root, Árvore Geradora e Papéis de Porta
Implementamos `executar_rstp(grafo)` reproduzindo integralmente:
1. Eleição do *Root Bridge* pelo menor `BID = (priority, mac)`;
2. Cálculo determinístico do menor *Root Path Cost* (desempate pelo menor BID do switch vizinho);
3. Classificação de cada porta em **Root Port (RP)**, **Designated Port (DP)** ou **Alternate Port (AP — Discarding)**;
4. Auditoria formal em Teoria dos Grafos provando que o subgrafo lógico $T = (V, E_T)$ é conexo, acíclico e possui $|E_T| = |V| - 1$.

In [ ]:
def eh_conexo(nos_lista, adj_map):
    if not nos_lista:
        return True
    visit = set([nos_lista[0]])
    q = deque([nos_lista[0]])
    while q:
        u = q.popleft()
        for v in adj_map[u]:
            if v not in visit:
                visit.add(v)
                q.append(v)
    return len(visit) == len(nos_lista)

def executar_rstp(grafo):
    """Simula a convergência do protocolo RSTP (IEEE 802.1w) sobre o grafo L2."""
    nos = grafo["nodes"]
    adj = grafo["adj"]
    lista_nos = list(nos.keys())

    if not eh_conexo(lista_nos, adj):
        raise ValueError("ALERTA CRÍTICO: O grafo da rede está particionado (desconexo)!")

    # Passo 1: Eleição do Root Bridge (menor tupla (priority, mac))
    root_bridge = min(lista_nos, key=lambda n: (nos[n]["priority"], nos[n]["mac"]))

    # Passo 2: Root Path Cost mínimo e escolha determinística da Root Port (RP)
    root_path_cost = {n: math.inf for n in lista_nos}
    root_port_neighbor = {n: None for n in lista_nos}
    root_path_cost[root_bridge] = 0

    visitados = set()
    while len(visitados) < len(lista_nos):
        u = min(
            (n for n in lista_nos if n not in visitados),
            key=lambda n: (root_path_cost[n], nos[n]["priority"], nos[n]["mac"])
        )
        visitados.add(u)

        for v, data in adj[u].items():
            if v in visitados:
                continue
            novo_custo = root_path_cost[u] + data["cost"]
            atual_pai = root_port_neighbor[v]
            bid_u = (nos[u]["priority"], nos[u]["mac"])
            bid_atual = (nos[atual_pai]["priority"], nos[atual_pai]["mac"]) if atual_pai else (math.inf, "")

            if (novo_custo < root_path_cost[v]) or (novo_custo == root_path_cost[v] and bid_u < bid_atual):
                root_path_cost[v] = novo_custo
                root_port_neighbor[v] = u

    # Passo 3: Papéis e estados das portas
    arestas_ativas = []
    arestas_bloqueadas = []
    tabela_portas = []

    for u, v, data in grafo["edges"]:
        if root_port_neighbor[v] == u:
            papel_u, estado_u = "Designated Port (DP)", "Forwarding"
            papel_v, estado_v = "Root Port (RP)", "Forwarding"
            arestas_ativas.append((u, v))
            status = "ATIVO (Árvore T)"
        elif root_port_neighbor[u] == v:
            papel_u, estado_u = "Root Port (RP)", "Forwarding"
            papel_v, estado_v = "Designated Port (DP)", "Forwarding"
            arestas_ativas.append((u, v))
            status = "ATIVO (Árvore T)"
        else:
            crit_u = (root_path_cost[u], nos[u]["priority"], nos[u]["mac"])
            crit_v = (root_path_cost[v], nos[v]["priority"], nos[v]["mac"])
            if crit_u < crit_v:
                papel_u, estado_u = "Designated Port (DP)", "Forwarding"
                papel_v, estado_v = "Alternate Port (AP)", "Discarding"
            else:
                papel_u, estado_u = "Alternate Port (AP)", "Discarding"
                papel_v, estado_v = "Designated Port (DP)", "Forwarding"
            arestas_bloqueadas.append((u, v))
            status = "BLOQUEADO (Loop Prevention)"

        tabela_portas.append({
            "Enlace": f"{u} <--> {v}",
            "Custo": data["cost"],
            "Ponta_A": f"{u}: {papel_u} [{estado_u}]",
            "Ponta_B": f"{v}: {papel_v} [{estado_v}]",
            "Status": status,
        })

    return {
        "root": root_bridge,
        "costs": root_path_cost,
        "parents": root_port_neighbor,
        "active_edges": arestas_ativas,
        "blocked_edges": arestas_bloqueadas,
        "table": tabela_portas,
    }

def auditar_arvore_geradora(grafo, arestas_ativas, titulo=""):
    """Comprova formalmente que T = (V, E_ativas) é uma Árvore Geradora."""
    lista_nos = list(grafo["nodes"].keys())
    adj_t = {n: [] for n in lista_nos}
    for u, v in arestas_ativas:
        adj_t[u].append(v)
        adj_t[v].append(u)

    n_v = len(lista_nos)
    n_e = len(arestas_ativas)
    conexo = eh_conexo(lista_nos, adj_t)
    edge_ok = (n_e == n_v - 1)
    e_arvore = conexo and edge_ok  # Teorema: conexo com |E| = |V| - 1 <=> acíclico (árvore)

    print(f"=== AUDITORIA FORMAL DA ÁRVORE GERADORA T {titulo} ===")
    print(f"  1. |V_T| == |V_G|          : True ({n_v} switches)")
    print(f"  2. |E_T| == |V_T| - 1      : {edge_ok} ({n_e} enlaces ativos)")
    print(f"  3. Conexo (sem ilhas)      : {conexo}")
    print(f"  4. Acíclico (0 Broadcast)  : {e_arvore}")
    print(f"  5. T é Árvore Geradora?    : {e_arvore}")
    if HAS_NX:
        T_nx = nx.Graph()
        T_nx.add_nodes_from(lista_nos)
        T_nx.add_edges_from(arestas_ativas)
        assert nx.is_tree(T_nx) == e_arvore
    return {"nodes": n_v, "edges": n_e, "connected": conexo, "is_tree": e_arvore}

def imprimir_tabela(tabela):
    if HAS_PD:
        print(pd.DataFrame(tabela).to_string(index=False))
    else:
        print(f"{'Enlace':27s} | {'Custo':>7s} | {'Status':28s} | Portas")
        print("-" * 115)
        for row in tabela:
            print(f"{row['Enlace']:27s} | {row['Custo']:7d} | {row['Status']:28s} | {row['Ponta_A']}  ||  {row['Ponta_B']}")

# Execução nominal
res_nominal = executar_rstp(G_nominal)
print(f"=== OPERAÇÃO NOMINAL — Root Bridge Eleito: {res_nominal['root']} ===")
for sw, c in res_nominal["costs"].items():
    print(f"  - {sw:10s} ({G_nominal['nodes'][sw]['setor']:34s}): Root Path Cost = {c}")
print()
imprimir_tabela(res_nominal["table"])
print()
aud_nominal = auditar_arvore_geradora(G_nominal, res_nominal["active_edges"], "(Nominal)")

assert res_nominal["root"] == "SW-SCADA"
assert res_nominal["costs"] == {"SW-SCADA": 0, "SW-PLC": 2000, "SW-IO-IN": 22000, "SW-IO-PROC": 40000, "SW-IO-OUT": 20000}
assert len(res_nominal["active_edges"]) == 4 and len(res_nominal["blocked_edges"]) == 2
assert aud_nominal["is_tree"] is True

### 3. Simulação de Falha 1 — Rompimento do Backbone `SW-SCADA <-> SW-PLC` (10 Gbps)
Simulamos a queda física do enlace principal entre o supervisório SCADA e o painel central `PLC-001`. O RSTP promove automaticamente a porta *Alternate* em `SW-IO-IN <-> SW-IO-PROC` para *Forwarding* em menos de $1\text{ s}$, preservando o *takt* nominal de $3{,}6\text{ s/copo}$ sem perda de supervisão.

In [ ]:
enlaces_falha1 = [(u, v, d) for (u, v, d) in ENLACES if {u, v} != {"SW-SCADA", "SW-PLC"}]
G_falha1 = construir_grafo(SWITCHES, enlaces_falha1)

res_falha1 = executar_rstp(G_falha1)
print("=== FALHA 1: ROMPIMENTO DO BACKBONE SW-SCADA <--> SW-PLC (10 Gbps) ===")
print(f"Root Bridge Mantido: {res_falha1['root']}")
for sw, c in res_falha1["costs"].items():
    print(f"  - {sw:10s}: Novo Custo = {c:6d} (Anterior: {res_nominal['costs'][sw]})")
print()
imprimir_tabela(res_falha1["table"])
print()
aud_falha1 = auditar_arvore_geradora(G_falha1, res_falha1["active_edges"], "(Após Falha 1)")

assert res_falha1["root"] == "SW-SCADA"
assert res_falha1["costs"]["SW-PLC"] == 80000
assert res_falha1["costs"]["SW-IO-IN"] == 60000
assert res_falha1["costs"]["SW-IO-PROC"] == 40000
assert res_falha1["costs"]["SW-IO-OUT"] == 20000
assert ("SW-IO-IN", "SW-IO-PROC") in res_falha1["active_edges"]
assert ("SW-PLC", "SW-IO-PROC") in res_falha1["blocked_edges"]
assert aud_falha1["is_tree"] is True

### 4. Simulação de Falha 2 — Queda Total do Switch Central `SW-SCADA`
Comprovamos que a configuração determinística de prioridades (`SW-PLC` com `Priority = 8192`) faz com que o painel do CLP assuma imediatamente como **Root Bridge Secundário**, mantendo todas as remotas de I/O (`SW-IO-IN`, `SW-IO-PROC`, `SW-IO-OUT`) operantes e sem ciclos enquanto a sala SCADA é restabelecida.

In [ ]:
switches_sem_scada = {k: v for k, v in SWITCHES.items() if k != "SW-SCADA"}
G_falha2 = construir_grafo(switches_sem_scada, ENLACES)

res_falha2 = executar_rstp(G_falha2)
print("=== FALHA 2: QUEDA TOTAL DO SW-SCADA (PROMOÇÃO DO ROOT SECUNDÁRIO) ===")
print(f"Novo Root Bridge Eleito: {res_falha2['root']} (Priority: {switches_sem_scada[res_falha2['root']]['priority']})")
for sw, c in res_falha2["costs"].items():
    print(f"  - {sw:10s}: Root Path Cost = {c}")
print()
aud_falha2 = auditar_arvore_geradora(G_falha2, res_falha2["active_edges"], "(Após Falha 2 — Célula Autônoma)")

assert res_falha2["root"] == "SW-PLC"
assert res_falha2["costs"] == {"SW-PLC": 0, "SW-IO-IN": 20000, "SW-IO-PROC": 40000, "SW-IO-OUT": 60000}
assert aud_falha2["nodes"] == 4 and aud_falha2["edges"] == 3 and aud_falha2["is_tree"] is True

### 5. Impacto no *Takt Time* da Envasadora (STP vs. RSTP) e Visualização Gráfica
Comparação quantitativa do número de ciclos de envase sem supervisão SCADA para um *takt* nominal de $3{,}6\text{ s/copo}$, seguida pela plotagem da topologia nominal vs. reconvergida (quando `matplotlib` + `networkx` estão disponíveis no ambiente Jupyter/Colab).

In [ ]:
TAKT_S = 3.6  # segundos por copo (Aqua Index — Aula 12)
cenarios_conv = [
    ("STP Clássico (IEEE 802.1D) — mínimo", 30.0),
    ("STP Clássico (IEEE 802.1D) — pior caso", 50.0),
    ("RSTP (IEEE 802.1w) — típico", 0.20),
    ("RSTP (IEEE 802.1w) — limite superior", 1.00),
]

print(f"=== IMPACTO DE CONVERGÊNCIA NO TAKT DA AQUA INDEX ({TAKT_S:.1f} s/copo) ===")
for nome, t_conv in cenarios_conv:
    ciclos_cegos = t_conv / TAKT_S
    print(f"  - {nome:42s}: {t_conv:5.2f} s  ->  {ciclos_cegos:5.2f} ciclos sem telemetria SCADA")

assert (30.0 / TAKT_S) > 8.0 and (0.20 / TAKT_S) < 0.1

if HAS_MPL and HAS_NX:
    def plotar_rstp(grafo_dict, res, titulo):
        pos = {
            "SW-SCADA":   (0.50, 1.00),
            "SW-PLC":     (0.12, 0.58),
            "SW-IO-OUT":  (0.88, 0.58),
            "SW-IO-IN":   (0.20, 0.10),
            "SW-IO-PROC": (0.80, 0.10),
        }
        G_nx = nx.Graph()
        for k, v in grafo_dict["nodes"].items():
            G_nx.add_node(k, **v)
        for u, v, d in grafo_dict["edges"]:
            G_nx.add_edge(u, v, **d)

        fig, ax = plt.subplots(figsize=(10.5, 5.6))
        cores = ["#F4C430" if n == res["root"] else "#2E6F95" for n in G_nx.nodes]
        nx.draw_networkx_nodes(G_nx, pos, node_color=cores, node_size=2600, edgecolors="#142534", linewidths=2, ax=ax)
        nx.draw_networkx_labels(G_nx, pos, font_size=9.5, font_weight="bold", font_color="#101a21", ax=ax)
        nx.draw_networkx_edges(G_nx, pos, edgelist=res["active_edges"], width=3.5, edge_color="#1F8A5B", ax=ax, label="Forwarding (Árvore T)")
        nx.draw_networkx_edges(G_nx, pos, edgelist=res["blocked_edges"], width=2.4, edge_color="#C62F24", style="dashed", ax=ax, label="Discarding (Alternate Port)")
        labels_e = {(u, v): f"w = {d['cost']}" for u, v, d in G_nx.edges(data=True)}
        nx.draw_networkx_edge_labels(G_nx, pos, edge_labels=labels_e, font_size=8.5, ax=ax)
        ax.set_title(f"Topologia Lógica RSTP — Aqua Index ({titulo}) | Root: {res['root']}", fontweight="bold")
        ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.08), ncol=2)
        ax.axis("off")
        plt.tight_layout()
        plt.show()

    plotar_rstp(G_nominal, res_nominal, "Operação Nominal")
    plotar_rstp(G_falha1, res_falha1, "Após Falha no Backbone SW-SCADA <-> SW-PLC")
else:
    print("\n[Info] Ambiente local sem matplotlib/networkx — validação numérica concluída; execute no Jupyter/Colab para gráficos.")

print("\n✅ Todas as validações RSTP (Nominal, Falha 1, Falha 2 e Takt) passaram com sucesso!")